In [1]:
# S4 — Deriving scaling factors from the spatial geometry
#
# The solenoid is a spatial structure. Influx enters and creates a pattern.
# At each position (coprime crossing), the 210 branches form a j3-comb
# whose spacing = 2*pi * e^{-kappa*ci}.  When spacing > pi, sheets wrap.
#
# The CP ratio at the CP pair (ci=11 vs ci=191) encodes the wrapping contrast.
# The mass exponent converts this to a mass ratio.
# The scaling factors r_bs and r_tc should emerge from the spatial geometry
# of the generation crossings.
#
# Strategy: Don't use PDG masses at all. Compute everything from the solenoid
# spatial structure and see what mass ratios emerge.

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi
lambda_P4 = reduce(_lcm, [p - 1 for p in primes])
phi_P4 = 48

P = [1]
for p in primes:
    P.append(P[-1] * p)

sys0 = SolenoidSystem(primes=primes)
all_branches = sys0.all_branches()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()
t_eval = cis.astype(float)
res = sys0.integrate_all_branches(all_branches, t_eval, float(P4 + 1), backend='jax')

# Compute RMS at all crossings, all levels
rms = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in all_branches])
        Rk_w = np.mod(Rk, 2 * np.pi)
        Rk_w[Rk_w > np.pi] -= 2 * np.pi
        rms[idx, k] = np.sqrt(np.mean(Rk_w**2))

gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}

def get_idx(a3v, a7v):
    return np.where((a3 == a3v) & (a5 == 0) & (a7 == a7v))[0][0]

# ══════════════════════════════════════════════════════════════
# The spatial picture: each crossing has a POSITION and an AMPLITUDE.
# The mass ratio between two fermions should come from the SPATIAL
# RELATIONSHIP between their crossings — not from fitting to PDG.
# ══════════════════════════════════════════════════════════════

print('=== The spatial influx picture ===')
print(f'The solenoid has P4={P4} positions. Influx enters and creates a wave.')
print(f'At each position ci, the 7 j3-sheets form a comb with spacing:')
print(f'  delta(ci) = 2*pi * e^(-kappa*ci)')
print(f'When delta > pi, sheets wrap. The wrapping boundary is where delta = pi:')
ci_boundary = -np.log(0.5) / kappa  # where e^{-kappa*ci} = 1/2, so delta = pi
print(f'  ci_boundary = ln(2)/kappa = {ci_boundary:.2f}')
print(f'  This is exactly the half-life! The wrapping boundary IS the spatial half-life.')

# ══════════════════════════════════════════════════════════════
# KEY IDEA: The mass ratio between two fermions is determined by
# the LOG-RATIO of the spatial amplitude AT THEIR CROSSINGS.
# But not the RMS directly — the RMS^x_q ratio.
#
# What if x_q itself comes from the spatial geometry?
# x_q = 1.587 at R3. What spatial quantity gives 1.587?
#
# The j3-comb at ci=11 has 7 sheets. 6 wrap, 1 doesn't.
# The WRAPPED sheets contribute RMS ≈ pi/sqrt(3) (uniform on [-pi,pi]).
# The UNWRAPPED sheet contributes its coherent value.
# The total RMS depends on the MIXTURE of wrapped and unwrapped.
#
# What if we can compute the CP ratio ANALYTICALLY from the comb structure?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Analytic comb model ===')

# At position ci on the solenoid, the j3-comb gives R3 values:
#   R3(j3, ci) = R3_base(ci) + j3 * 2*pi * e^{-kappa*ci}
# where R3_base is the j3=0 branch value.

# The wrapped RMS^2 for a uniform distribution on [-pi, pi] = pi^2/3
rms_uniform = np.pi / np.sqrt(3)
print(f'RMS of uniform [-pi,pi]: pi/sqrt(3) = {rms_uniform:.4f}')

# At ci=11: spacing = 2*pi*e^{-kappa*11} = 2*pi*0.468 = 2.941
spacing_11 = 2 * np.pi * np.exp(-kappa * 11)
print(f'Comb spacing at ci=11: {spacing_11:.4f} (compare pi={np.pi:.4f})')

# At ci=191: spacing = 2*pi*e^{-kappa*191} = essentially 0
spacing_191 = 2 * np.pi * np.exp(-kappa * 191)
print(f'Comb spacing at ci=191: {spacing_191:.6e} (all sheets collapsed)')

# For ci=191, all sheets are at essentially the same value.
# R3(j3=0..6, ci=191) ≈ R3_base(191) for all j3.
# So RMS(191) = |R3_base(191)|.

# For ci=11, 6/7 sheets wrap. The wrapped RMS depends on how
# the comb teeth distribute modulo 2*pi.

# Let me compute the EXACT comb model at each quark generation crossing
print(f'\n=== Comb model vs actual cascade at quark Z2=0 crossings ===')

# Get actual branch data
for ci_val in [11, 71, 191]:
    ci_idx = np.where(cis == ci_val)[0][0]
    R3_branches = np.array([res[br][ci_idx, 3] for br in all_branches])
    
    # Group by j3
    j3_vals = np.array([br[3] for br in all_branches])
    j3_means = [np.mean(R3_branches[j3_vals == j]) for j in range(7)]
    
    # The comb: base value and spacing
    base = j3_means[0]
    if len(j3_means) > 1:
        spacing = np.mean(np.diff(j3_means))
    else:
        spacing = 0
    
    # Predicted spacing from kappa
    predicted_spacing = 2 * np.pi * np.exp(-kappa * ci_val)
    
    # Compute wrapped RMS from the comb model
    comb_values = np.array([base + j * spacing for j in range(7)])
    comb_wrapped = np.mod(comb_values, 2*np.pi)
    comb_wrapped[comb_wrapped > np.pi] -= 2*np.pi
    # Each j3 group has P3=30 branches with small spread
    # So total RMS ≈ RMS of the 7 comb values (each weighted by 30)
    comb_rms = np.sqrt(np.mean(comb_wrapped**2))
    actual_rms = rms[ci_idx, 3]
    
    gen = {11: 2, 71: 1, 191: 3}[ci_val]
    print(f'  ci={ci_val:3d} gen{gen}: base={base:.4f} spacing={spacing:.4f} '
          f'(pred={predicted_spacing:.4f}) comb_rms={comb_rms:.4f} actual={actual_rms:.4f} '
          f'({abs(comb_rms-actual_rms)/actual_rms*100:.1f}%)')

# ══════════════════════════════════════════════════════════════
# The comb model gives us CP ratios as a function of ci positions.
# Now: can we compute mass ratios from PAIRS of crossings?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Computing CP ratios between ALL generation pairs ===')
# For each pair of generation crossings, compute the CP ratio at each level
# The mass ratio should be CP^{x_q} for 1->2 step, CP^{x_q*r} for 2->3 step.

# All 6 quark crossings (a3=1, a5=0)
q_data = {}
for a7v in range(6):
    idx = get_idx(1, a7v)
    q_data[a7v] = {'ci': cis[idx], 'rms': rms[idx], 'gen': gen_map[a7v], 'z2': a7v % 2}

# Cross-generation CP ratios within Z2=0 coset
print(f'\nZ2=0 quark cross-generation ratios:')
g1 = q_data[0]  # gen1 Z2=0, ci=71
g2 = q_data[4]  # gen2 Z2=0, ci=11
g3 = q_data[2]  # gen3 Z2=0, ci=191

pairs = [
    ('gen2/gen3 (CP pair)', g2, g3, 'CP pair: m_s/m_d channel'),
    ('gen1/gen3', g1, g3, 'spanning gen1 to gen3'),
    ('gen2/gen1', g2, g1, 'gen2 over gen1'),
]

x_q = 1.5866463961
for label, ga, gb, desc in pairs:
    print(f'\n  {label} ({desc}):')
    for k in range(4):
        ratio = ga['rms'][k] / gb['rms'][k]
        mass_ratio = ratio ** x_q if ratio > 0.001 else 0
        # Also compute with factored exponent at this level
        cp_pair_ratio = g2['rms'][k] / g3['rms'][k]
        if cp_pair_ratio > 1.001:
            x_factored = x_q * np.log(g2['rms'][3]/g3['rms'][3]) / np.log(cp_pair_ratio)
            mass_f = ratio ** x_factored
        else:
            x_factored = 0
            mass_f = 0
        print(f'    R{k}: ratio={ratio:.4f} ratio^x_q={mass_ratio:.2f}')

# ══════════════════════════════════════════════════════════════
# CRITICAL TEST: Do the mass ratios come from cross-generation
# CP ratios computed FROM THE SPATIAL STRUCTURE?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Mass ratios from spatial cross-generation structure ===')

# The CP pair (gen2/gen3) at R3 gives CP=6.607.
# CP^{x_q} = 20.0 = m_s/m_d. This is the 1->2 down-type ratio.
# 
# But m_s/m_d is gen2/gen1 physically. The CP pair is gen2/gen3.
# So CP(gen2/gen3)^x_q gives the gen2/gen1 MASS ratio.
# This means the CP pair and the mass channel are DIFFERENT generation comparisons.
#
# What if we compute CP for EACH generation pair?
# CP(gen1/gen3) at R3 = 2.096
# CP(gen2/gen1) at R3 = 3.152

# Question: does CP(gen1/gen3)^{x_q} or CP(gen2/gen1)^{x_q} give any known mass ratio?
r_g1 = g1['rms']  # ci=71
r_g2 = g2['rms']  # ci=11
r_g3 = g3['rms']  # ci=191

cp_23 = r_g2[3] / r_g3[3]  # gen2/gen3 = 6.607
cp_13 = r_g1[3] / r_g3[3]  # gen1/gen3 = 2.096
cp_21 = r_g2[3] / r_g1[3]  # gen2/gen1 = 3.152

print(f'Cross-generation CP at R3:')
print(f'  gen2/gen3 = {cp_23:.4f}')
print(f'  gen1/gen3 = {cp_13:.4f}')
print(f'  gen2/gen1 = {cp_21:.4f}')
print(f'  Check: (gen2/gen1)*(gen1/gen3) = {cp_21*cp_13:.4f} = gen2/gen3? {abs(cp_21*cp_13 - cp_23) < 0.001}')

# Now raise each to x_q:
print(f'\nRaised to x_q = {x_q:.4f}:')
print(f'  (gen2/gen3)^x_q = {cp_23**x_q:.2f}  ← this is m_s/m_d = 20.0')
print(f'  (gen1/gen3)^x_q = {cp_13**x_q:.4f}')
print(f'  (gen2/gen1)^x_q = {cp_21**x_q:.2f}')

# The gen2/gen3 ratio raised to x_q gives m_s/m_d.
# What about (gen2/gen3)^x_q * (gen2/gen1)^x_q = (gen2^2/(gen1*gen3))^x_q?
cross_product = (cp_23 * cp_21) ** x_q
print(f'\n  (gen2/gen3 * gen2/gen1)^x_q = (gen2^2/(gen1*gen3))^x_q = {cross_product:.2f}')
print(f'  Compare: m_b/m_s * m_s/m_d = m_b/m_d = 44.75 * 20 = 895')

# Or: (gen2/gen3)^x_q * (gen1/gen3)^x_q = (gen2*gen1/gen3^2)^x_q
cross_product2 = (cp_23 * cp_13) ** x_q
print(f'  (gen2/gen3 * gen1/gen3)^x_q = (gen1*gen2/gen3^2)^x_q = {cross_product2:.2f}')

# The COMPLETE mass information might come from the FULL spatial profile
# at all three generation crossings, not just the CP pair.
# Let me compute what mass ratios emerge from ALL possible combinations.

print(f'\n=== All possible mass ratios from spatial crossings ===')
# There are three independent RMS values: R3(g1), R3(g2), R3(g3)
# And two independent ratios: gen2/gen3 and gen1/gen3 (or gen2/gen1)
# These determine ALL possible ratios.

# The mass hierarchy has 5 independent down-type ratios:
# m_s/m_d (gen2/gen1), m_b/m_s (gen3/gen2), m_b/m_d (gen3/gen1)
# Plus the up-type versions.
# From 2 independent cross-gen ratios, we can get at most 2 independent mass ratios.
# But the CP pair gives ONE ratio, and x_q converts it to ONE mass ratio.
# The OTHER mass ratios need ADDITIONAL information.

# The additional information is: the gen1 crossing amplitude.
# gen1 is at ci=71 with R3=0.5858.
# This is a THIRD data point that the CP pair doesn't use.

# What if the mass hierarchy comes from ALL THREE generation amplitudes?
# mass(gen_k) ∝ some function of R3(gen_k)
# Then mass ratios = function ratios.

# Test: mass ∝ R3^{-x_q}? (inverse because heavy gen has small R3)
# m_b/m_s = [R3(gen2)/R3(gen3)]^{x_q} = CP^{x_q} = 20.0
# But m_b/m_s = 44.75, not 20.0. So this doesn't work for 2->3 step.

# What about mass ∝ exp(-alpha * R3)?
# Or mass ∝ 1/R3^n for some n?

# Let me try a different approach: compute mass from the POSITION on the solenoid.
# mass(gen_k) ∝ exp(beta * ci_k) for some beta?
ci_g1, ci_g2, ci_g3 = 71, 11, 191

# If mass ∝ exp(beta * ci), then:
# m_s/m_d = exp(beta*(ci_g2 - ci_g1)) if gen2 maps to s, gen1 to d
# But which gen maps to which quark?
# Heavy gen3 → b (heaviest), gen2 → s or c, gen1 → d or u

# m_b is gen3 (heaviest), m_s is gen2, m_d is gen1.
# In the solenoid, ci_g3 = 191 (furthest from influx source, lowest amplitude)
# This is where influx is weakest → gives the heaviest fermion.
# BECAUSE: mass comes from covering MISALIGNMENT. Where influx is strong (ci=11),
# the misalignment is large but gets WRAPPED → modular reduction.
# Where influx is weak (ci=191), the misalignment is small but COHERENT.
# The coherent small value → light fermion. The wrapped large value → heavy fermion?
# No, heavy fermion = gen3 = ci=191 = SMALLEST RMS. 

# Actually: maybe the mass is not from the amplitude but from the PHASE COHERENCE.
# Heavy fermion = most phase-coherent (least wrapping), which means the covering
# structure at that crossing is most tightly aligned → most "mass" (resistance to change).

# Let me try: the ci positions form an arithmetic sequence mod P4.
# gen2=11, gen1=71, gen3=191. Spacings: 60, 120 (or 30, 60 going the other way)
# On the solenoid circle, these are at phases: 11/210, 71/210, 191/210

# What if mass ratio = function of phase difference between crossings?
# The phase difference between gen2 and gen3: (191-11)/210 = 180/210 = 6/7
# The phase difference between gen1 and gen2: (71-11)/210 = 60/210 = 2/7
# The phase difference between gen1 and gen3: (191-71)/210 = 120/210 = 4/7

print(f'\n=== Phase differences on the solenoid circle ===')
print(f'gen2→gen3: {(ci_g3-ci_g2)/P4:.4f} = {ci_g3-ci_g2}/{P4} = {(ci_g3-ci_g2)//30}/7')
print(f'gen1→gen2: {(ci_g1-ci_g2)/P4:.4f} = {ci_g1-ci_g2}/{P4} = {(ci_g1-ci_g2)//30}/7')
print(f'gen1→gen3: {(ci_g3-ci_g1)/P4:.4f} = {ci_g3-ci_g1}/{P4} = {(ci_g3-ci_g1)//30}/7')

# Phase differences in units of 1/7:
# gen2→gen3: 6/7  (or equivalently -1/7 going backward: 30/210 = 1/7)
# gen1→gen2: 2/7
# gen1→gen3: 4/7

# These are MULTIPLES OF 1/7 = 1/p4!
# Because the generation structure comes from Z6, and the CRT maps 
# Z6 steps to ci-steps of P4/7 = 30 = P3.

# Now: the influx pattern decays as e^{-kappa*ci}. The amplitude RATIO
# between two points separated by delta_ci is e^{-kappa*delta_ci}.

# For the CP pair (gen2→gen3, delta=180):
# e^{-kappa*180} = e^{-180/sqrt(210)} = e^{-12.42} = 4.0e-6
# This is the ratio of the DOMINANT MODE amplitudes.
# But the actual CP ratio is 6.607, vastly different from 4e-6!

# The difference is wrapping. The dominant mode at ci=11 is 2.46 (from S2 fit)
# but 85.7% of branches wrap, creating an RMS of 1.847.
# At ci=191, no wrapping, RMS = 0.280.
# The CP ratio = 1.847/0.280 = 6.607 is a NONLINEAR FUNCTION of the
# dominant mode amplitudes, mediated by wrapping.

# So: the spatial influx creates a smooth exponential pattern.
# The 7-sheet comb converts this smooth pattern into a DISCRETE wrapping structure.
# The wrapping is a NONLINEAR OPERATION on the smooth influx.
# Mass comes from this nonlinear transformation, not from the smooth pattern.

print(f'\n=== The nonlinear mechanism: influx → comb → wrapping → mass ===')
print(f'1. Influx creates smooth exponential: A(ci) = A0 * e^(-kappa*ci)')
print(f'2. p4=7 sheets create comb: R3(j3, ci) = base(ci) + j3 * delta(ci)')
print(f'   where delta(ci) = 2*pi * e^(-kappa*ci)')
print(f'3. Wrapping: R3 mod 2*pi maps the comb onto [-pi, pi]')
print(f'4. RMS of the wrapped comb = the observable spatial amplitude')
print(f'5. CP ratio = RMS(wrapping zone) / RMS(coherent zone)')
print(f'6. Mass = CP^x_q where x_q is the spatial coupling eigenvalue')

# Can we compute the comb RMS analytically as a function of ci?
# If the comb values are: base + j*delta for j=0,...,6
# wrapped to [-pi, pi], what's the RMS?

def comb_rms_analytic(base, delta, n_teeth=7):
    """RMS of n_teeth equally spaced values, wrapped to [-pi, pi]."""
    values = np.array([base + j * delta for j in range(n_teeth)])
    wrapped = np.mod(values, 2*np.pi)
    wrapped[wrapped > np.pi] -= 2*np.pi
    return np.sqrt(np.mean(wrapped**2))

# Compute comb RMS as a function of ci
print(f'\n=== Comb RMS vs ci position ===')
# Use actual base values from cascade
ci_test = [1, 11, 31, 41, 71, 101, 131, 191]
for ci_val in ci_test:
    ci_idx = np.where(cis == ci_val)[0]
    if len(ci_idx) == 0:
        continue
    ci_idx = ci_idx[0]
    
    # Get actual j3=0 branch value at R3
    j3_vals = np.array([br[3] for br in all_branches])
    R3_all = np.array([res[br][ci_idx, 3] for br in all_branches])
    base_actual = np.mean(R3_all[j3_vals == 0])
    delta_actual = 2 * np.pi * np.exp(-kappa * ci_val)
    
    comb_rms_val = comb_rms_analytic(base_actual, delta_actual)
    actual_rms_val = rms[ci_idx, 3]
    
    gen = gen_map.get(a7[ci_idx], '?')
    tag = ''
    if ci_val == 11: tag = ' ← gen2 (CP g1)'
    elif ci_val == 71: tag = ' ← gen1'
    elif ci_val == 191: tag = ' ← gen3 (CP g2)'
    
    print(f'  ci={ci_val:3d}: base={base_actual:.4f} delta={delta_actual:.4f} '
          f'comb_rms={comb_rms_val:.4f} actual={actual_rms_val:.4f} '
          f'({abs(comb_rms_val-actual_rms_val)/actual_rms_val*100:.1f}%){tag}')

# ══════════════════════════════════════════════════════════════
# THE KEY: can we compute CP ratios and mass ratios from the comb model?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Mass ratios from pure comb geometry ===')
# Get base values at each generation crossing
ci_gen = {1: 71, 2: 11, 3: 191}  # gen -> ci for Z2=0 quarks
bases = {}
for gen_id, ci_val in ci_gen.items():
    ci_idx = np.where(cis == ci_val)[0][0]
    j3_vals_arr = np.array([br[3] for br in all_branches])
    R3_all = np.array([res[br][ci_idx, 3] for br in all_branches])
    bases[gen_id] = np.mean(R3_all[j3_vals_arr == 0])

# Compute comb RMS at each gen crossing
comb_rms_gen = {}
for gen_id, ci_val in ci_gen.items():
    delta = 2 * np.pi * np.exp(-kappa * ci_val)
    comb_rms_gen[gen_id] = comb_rms_analytic(bases[gen_id], delta)

print(f'Comb RMS: gen1={comb_rms_gen[1]:.6f} gen2={comb_rms_gen[2]:.6f} gen3={comb_rms_gen[3]:.6f}')
print(f'Actual:   gen1={rms[get_idx(1,0),3]:.6f} gen2={rms[get_idx(1,4),3]:.6f} gen3={rms[get_idx(1,2),3]:.6f}')

# CP from comb model
cp_comb = comb_rms_gen[2] / comb_rms_gen[3]
print(f'\nComb CP(gen2/gen3) = {cp_comb:.4f} (actual = {cp_23:.4f})')

# All three cross-gen ratios from comb:
cp_comb_23 = comb_rms_gen[2] / comb_rms_gen[3]
cp_comb_13 = comb_rms_gen[1] / comb_rms_gen[3]
cp_comb_21 = comb_rms_gen[2] / comb_rms_gen[1]

print(f'Comb gen2/gen3 = {cp_comb_23:.4f}')
print(f'Comb gen1/gen3 = {cp_comb_13:.4f}')
print(f'Comb gen2/gen1 = {cp_comb_21:.4f}')

# NOW: if we raise each to x_q, what mass ratios do we get?
print(f'\nComb ratios raised to x_q:')
print(f'  (gen2/gen3)^x_q = {cp_comb_23**x_q:.2f}  (PDG m_s/m_d = 20.0)')
print(f'  (gen1/gen3)^x_q = {cp_comb_13**x_q:.4f}')
print(f'  (gen2/gen1)^x_q = {cp_comb_21**x_q:.2f}')

# And the KEY test: do combinations of these give m_b/m_s and m_t/m_c?
# m_b/m_s = 44.75. m_t/m_c = 135.8.
# If the spatial structure determines everything, these should emerge.
print(f'\n--- Testing spatial mass predictions ---')
# m_b/m_d = m_b/m_s * m_s/m_d = 895 (PDG)
# gen1→gen3 spans BOTH generation steps.
# (gen2/gen3)^x_q * (something from gen1) = m_b/m_d?
print(f'  (gen2*gen1/gen3^2)^x_q = {((comb_rms_gen[2]*comb_rms_gen[1])/comb_rms_gen[3]**2)**x_q:.2f}')
print(f'  (gen2^2/(gen1*gen3))^x_q = {((comb_rms_gen[2]**2)/(comb_rms_gen[1]*comb_rms_gen[3]))**x_q:.2f}')
print(f'  (gen1/gen3)^x_q = {cp_comb_13**x_q:.2f}')

# What about using log-space?
# ln(m_b/m_d) = ln(m_b/m_s) + ln(m_s/m_d)
# In spatial terms: the full gen1→gen3 span should give m_b/m_d
# = (gen1→gen3 spatial ratio)^x_q? 
# NO — because the scaling factors r_bs ≠ 1. The 1→2 and 2→3 steps
# use different scaling on x_q.
#
# UNLESS the spatial picture gives us a DIFFERENT decomposition
# where we don't need separate scaling factors.
# What if mass = f(ci) where f depends only on the position?

# Test: ln(mass_k) = alpha * ci_k + beta
# m_d ∝ exp(alpha * 71), m_s ∝ exp(alpha * 11), m_b ∝ exp(alpha * 191)
# But gen3 (heaviest) at ci=191 (largest) and gen1 (lightest) at ci=71 (smaller)
# So alpha > 0 would make gen3 heaviest — which is correct!
# mass_k ∝ exp(alpha * ci_k) for the generation crossings.
# Then m_b/m_s = exp(alpha * (191-11)) = exp(180*alpha)
# And m_s/m_d = exp(alpha * (11-71)) = exp(-60*alpha)
# But m_s > m_d, so m_s/m_d > 1, which needs -60*alpha > 0, i.e. alpha < 0.
# Contradiction: alpha > 0 for gen3 heaviest but alpha < 0 for m_s > m_d.

# The problem: gen2 (ci=11) maps to s (lighter) while gen3 (ci=191) maps to b (heavier).
# But ci=11 < ci=191. So increasing ci → heavier fermion.
# Meanwhile gen1 (ci=71) maps to d (lightest). ci=71 is between 11 and 191.
# So ci ordering is 11 < 71 < 191 but mass ordering is d < s < b.
# Mass increases with ci? s(ci=11) vs d(ci=71): s > d and ci_s < ci_d. NO.

# The mapping is gen-based, not ci-based:
# gen1(ci=71) → d (lightest)
# gen2(ci=11) → s (middle)  
# gen3(ci=191) → b (heaviest)
# ci ordering: 11 < 71 < 191
# mass ordering: s > d < b (not monotonic in ci!)

# So mass is NOT a simple function of ci. It depends on the GENERATION
# identity, not the position. The position determines the RMS, and the
# mass exponent converts RMS ratios to mass ratios.

print(f'\n=== Summary: what the spatial picture gives us ===')
print(f'The solenoid spatial structure provides:')
print(f'  1. The CP pair ratio = {cp_23:.4f} (from wrapping at ci=11 vs coherence at ci=191)')
print(f'  2. This ratio^x_q = {cp_23**x_q:.2f} = m_s/m_d')
print(f'  3. The gen1 RMS = {comb_rms_gen[1]:.4f} is a THIRD independent data point')
print(f'  4. From (gen2/gen3) and (gen1/gen3), we get {cp_comb_13**x_q:.4f} = (gen1/gen3)^x_q')
print(f'')
print(f'The scaling factor r_bs = ln(m_b/m_s) / ln(m_s/m_d) = 1.269')
print(f'In spatial terms: r_bs = ln(m_b/m_s) / ln(CP^x_q)')
print(f'                      = ln(44.75) / ln(20.0) = 1.269')
print(f'')
print(f'Can we get this from the THREE generation amplitudes alone?')
print(f'  ln(gen2/gen3) / ln(gen2/gen3) = 1 (trivially)')  
print(f'  ln(gen1/gen3) / ln(gen2/gen3) = {np.log(cp_comb_13)/np.log(cp_comb_23):.6f}')
print(f'  Compare: 1 + r_bs = 1 + 1.269 = 2.269')
print(f'           ln(gen1/gen3)/ln(gen2/gen3) = {np.log(cp_comb_13)/np.log(cp_comb_23):.6f}')
print(f'  These are NOT equal.')
print(f'  The scaling factors encode MASS physics, not just spatial geometry.')

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.66s
=== The spatial influx picture ===
The solenoid has P4=210 positions. Influx enters and creates a wave.
At each position ci, the 7 j3-sheets form a comb with spacing:
  delta(ci) = 2*pi * e^(-kappa*ci)
When delta > pi, sheets wrap. The wrapping boundary is where delta = pi:
  ci_boundary = ln(2)/kappa = 10.04
  This is exactly the half-life! The wrapping boundary IS the spatial half-life.

=== Analytic comb model ===
RMS of uniform [-pi,pi]: pi/sqrt(3) = 1.8138
Comb spacing at ci=11: 2.9412 (compare pi=3.1416)
Comb spacing at ci=191: 1.185957e-05 (all sheets collapsed)

=== Comb model vs actual cascade at quark Z2=0 crossings ===
  ci= 11 gen2: base=0.8713 spacing=2.9412 (pred=2.9412) comb_rms=1.8712 actual=1.8465 (1.3%)
  ci= 71 gen1: base=0.4326 spacing=0.0468 (pred=0.0468) comb_rms=0.5807 actual=0.5858 (0.9%)
  ci=191 gen3: base=0.2795 spacing=0.0000 (pred=0.0000) comb_rms=0.2795 actual=0.2795 (0.0%)

=== Computin

In [1]:
# S3 — The wrapping transition: anatomy of ci=11
# 
# Only ONE crossing has wrapping (85.7% at ci=11). All mass information
# flows through this single spatial point. What determines the wrapping
# there, and how does it produce the mass spectrum?

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi

P = [1]
for p in primes:
    P.append(P[-1] * p)

sys0 = SolenoidSystem(primes=primes)
all_branches = sys0.all_branches()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()
t_eval = cis.astype(float)
res = sys0.integrate_all_branches(all_branches, t_eval, float(P4 + 1), backend='jax')

# ══════════════════════════════════════════════════════════════
# The 210 branches at ci=11: how are they distributed?
# ══════════════════════════════════════════════════════════════

ci_11_idx = np.where(cis == 11)[0][0]
print(f'=== Anatomy of ci=11 (gen2 Z2=0, the wrapping point) ===')

# R3 values for all 210 branches at ci=11
R3_at_11 = np.array([res[br][ci_11_idx, 3] for br in all_branches])
R3_wrapped = np.mod(R3_at_11, 2*np.pi)
R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi

print(f'210 branches at ci=11, level R3:')
print(f'  Raw R3 range: [{R3_at_11.min():.4f}, {R3_at_11.max():.4f}]')
print(f'  Wrapped R3 range: [{R3_wrapped.min():.4f}, {R3_wrapped.max():.4f}]')
print(f'  Wrapped RMS: {np.sqrt(np.mean(R3_wrapped**2)):.6f}')

# How many wrapping levels?
n_wraps = np.round(R3_at_11 / (2*np.pi)).astype(int)
print(f'\n  Wrapping distribution:')
for w in range(n_wraps.min(), n_wraps.max()+1):
    count = np.sum(n_wraps == w)
    if count > 0:
        print(f'    {w:+d} wraps: {count} branches ({count/210*100:.1f}%)')

# Now look at ALL FOUR LEVELS at ci=11
print(f'\n  All levels at ci=11:')
for k in range(4):
    Rk = np.array([res[br][ci_11_idx, k] for br in all_branches])
    n_w = np.sum(np.abs(Rk) > np.pi)
    print(f'    R{k}: range=[{Rk.min():.2f}, {Rk.max():.2f}] wrapped={n_w/210*100:.1f}%')

# ══════════════════════════════════════════════════════════════
# Compare ALL crossings: how does wrapping fraction vary spatially?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Wrapping fraction vs spatial position (all crossings) ===')
print(f'{"ci":>4}  {"wrap_R0":>8} {"wrap_R1":>8} {"wrap_R2":>8} {"wrap_R3":>8}  {"a3":>3} {"a5":>3} {"a7":>3}')
wrap_fracs = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in all_branches])
        wrap_fracs[idx, k] = np.sum(np.abs(Rk) > np.pi) / 210
    if wrap_fracs[idx].sum() > 0:  # only show crossings with some wrapping
        ci_val = cis[idx]
        print(f'{ci_val:4d}  {wrap_fracs[idx,0]*100:7.1f}% {wrap_fracs[idx,1]*100:7.1f}% '
              f'{wrap_fracs[idx,2]*100:7.1f}% {wrap_fracs[idx,3]*100:7.1f}%  '
              f'{a3[idx]:3d} {a5[idx]:3d} {a7[idx]:3d}')

# ══════════════════════════════════════════════════════════════
# The wrapping boundary: which branches wrap and why?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Branch anatomy at the wrapping boundary ===')
# The 210 branches have initial j-values (j0, j1, j2, j3).
# The R3 value at ci=11 depends on the initial conditions.
# Which j-values lead to wrapping?

# Each branch is indexed by (j0, j1, j2, j3) where j_k runs from 0 to p_{k+1}-1
# (i.e., j0 in 0..1, j1 in 0..2, j2 in 0..4, j3 in 0..6)
# The total number of branches = 2*3*5*7 = 210.

# At ci=11, the outermost level R3 = p4*theta3 - theta2 evaluated at the
# crossing point. The j3 initial condition (which sheet of the p4=7 covering)
# determines the primary R3 contribution.

# Group branches by j3 (the outermost IC):
j3_groups = {}
for br_idx, br in enumerate(all_branches):
    j3 = br[3]  # outermost IC
    if j3 not in j3_groups:
        j3_groups[j3] = []
    j3_groups[j3].append(R3_at_11[br_idx])

print(f'R3 at ci=11 grouped by j3 (outermost initial condition):')
for j3 in sorted(j3_groups.keys()):
    vals = np.array(j3_groups[j3])
    n_w = np.sum(np.abs(vals) > np.pi)
    print(f'  j3={j3}: n={len(vals)} mean={np.mean(vals):+8.4f} '
          f'std={np.std(vals):.4f} wrapped={n_w}/{len(vals)} '
          f'range=[{vals.min():.2f}, {vals.max():.2f}]')

# The j3 value determines the STARTING SHEET on the p=7 covering.
# Different starting sheets lead to different amounts of wrapping.
# The distribution of wrapping across j3 values IS the standing wave
# fine structure at this spatial point.

# ══════════════════════════════════════════════════════════════
# KEY TEST: Can we predict mass ratios from the spatial geometry?
# ══════════════════════════════════════════════════════════════

print(f'\n=== The spatial geometry of mass ===')
# The CP ratio at each level k is:
#   CP_Rk = RMS(ci=11, level k) / RMS(ci=191, level k)
# This is a ratio of spatial amplitudes at two points on the solenoid.
# The mass ratio = CP^x where x is the exponent.
# 
# In the spatial picture: the mass ratio is determined by the RATIO of
# the spatial wave amplitude at two generation crossings.
# But CP^x is not a simple amplitude ratio — it involves the exponent.
# What IS CP^x geometrically?
#
# ln(mass) = x * ln(CP) = x * [ln(RMS_g2) - ln(RMS_g3)]
# = x * [ln(wave at gen2) - ln(wave at gen3)]
# = the WEIGHTED LOG-DIFFERENCE of the spatial wave at two points.
# The weight x encodes the coupling between the wave and the mass degree of freedom.

# In the spatial picture, the wave amplitude at each point is:
# R3_RMS(ci) = f(ci, all branch ICs)
# This is a function of position on the solenoid and the full branch structure.
# The mass ratio extracts the LOG RATIO of this function at two specific points.

# Maybe the exponent x itself has spatial meaning:
# x = number of effective wrapping LAYERS contributing to the mass
# At ci=11, 85.7% of branches wrap. The wrapping creates multiple 2pi layers.
# The exponent might count these layers.

# Check: how many 2pi layers exist in R3 at ci=11?
print(f'Wrapping layers at ci=11, R3:')
R3_abs = np.abs(R3_at_11)
n_layers = np.floor(R3_abs / np.pi)
max_layers = int(n_layers.max())
for L in range(max_layers + 1):
    count = np.sum(n_layers == L)
    if count > 0:
        print(f'  {L} half-periods: {count} branches ({count/210*100:.1f}%)')

# Mean number of half-periods:
mean_hp = np.mean(n_layers)
print(f'\n  Mean half-periods at ci=11: {mean_hp:.4f}')
print(f'  x_q = {1.5866:.4f}')
print(f'  x_q / mean_hp = {1.5866/mean_hp:.4f}')

# At ci=191:
ci_191_idx = np.where(cis == 191)[0][0]
R3_at_191 = np.array([res[br][ci_191_idx, 3] for br in all_branches])
mean_hp_191 = np.mean(np.floor(np.abs(R3_at_191) / np.pi))
print(f'  Mean half-periods at ci=191: {mean_hp_191:.4f}')

# Try: x_q relates to the DIFFERENCE in wrapping structure between g2 and g3?
print(f'  Wrapping difference: {mean_hp - mean_hp_191:.4f}')
print(f'  Ratio of wrapping: {mean_hp / max(mean_hp_191, 0.001):.4f}')

# Per-level wrapping layer analysis
print(f'\n=== Wrapping layers at each level ===')
for k in range(4):
    Rk_11 = np.array([res[br][ci_11_idx, k] for br in all_branches])
    Rk_191 = np.array([res[br][ci_191_idx, k] for br in all_branches])
    hp_11 = np.mean(np.floor(np.abs(Rk_11) / np.pi))
    hp_191 = np.mean(np.floor(np.abs(Rk_191) / np.pi))
    rms_11 = np.sqrt(np.mean(np.mod(Rk_11, 2*np.pi).clip(-np.pi, np.pi)**2))
    rms_191 = np.sqrt(np.mean(np.mod(Rk_191, 2*np.pi).clip(-np.pi, np.pi)**2))
    cp = rms_11 / rms_191 if rms_191 > 0.001 else float('inf')
    print(f'  R{k}: ci=11 hp={hp_11:.2f} rms={rms_11:.4f}  ci=191 hp={hp_191:.2f} rms={rms_191:.4f}  CP={cp:.4f}')

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.38s
=== Anatomy of ci=11 (gen2 Z2=0, the wrapping point) ===
210 branches at ci=11, level R3:
  Raw R3 range: [0.4182, 19.2286]
  Wrapped R3 range: [-3.1316, 3.1180]
  Wrapped RMS: 1.846494

  Wrapping distribution:
    +0 wraps: 30 branches (14.3%)
    +1 wraps: 67 branches (31.9%)
    +2 wraps: 76 branches (36.2%)
    +3 wraps: 37 branches (17.6%)

  All levels at ci=11:
    R0: range=[-0.01, 2.94] wrapped=0.0%
    R1: range=[0.04, 6.99] wrapped=50.0%
    R2: range=[-0.01, 13.44] wrapped=73.3%
    R3: range=[0.42, 19.23] wrapped=85.7%

=== Wrapping fraction vs spatial position (all crossings) ===
  ci   wrap_R0  wrap_R1  wrap_R2  wrap_R3   a3  a5  a7
   1     50.0%    66.7%    80.0%    85.7%    0   0   0
  11      0.0%    50.0%    73.3%    85.7%    1   0   4
  13      0.0%    50.0%    73.3%    82.4%    0   3   3
  17      0.0%    33.3%    66.7%    73.3%    1   1   1
  19      0.0%    33.3%    66.7%    72.4%    0   2   

In [1]:
# S2 — The solenoid as a resonant spatial structure
#
# Reframe: the ODE parameter is a SPATIAL coordinate on the solenoid, not time.
# The solution R(ci) is a spatial profile — a standing wave of covering misalignment.
# The "overdamped" character means no oscillatory nodes: one dominant mode.
# kappa = 1/sqrt(P4) is the spatial decay length.
#
# Question: does the standing wave structure, sampled at the generation crossings,
# determine the scaling factors r_bs and r_tc?

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi
lambda_P4 = reduce(_lcm, [p - 1 for p in primes])

P = [1]
for p in primes:
    P.append(P[-1] * p)

# Run cascade (= compute spatial profile)
sys0 = SolenoidSystem(primes=primes)
all_branches = sys0.all_branches()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()
t_eval = cis.astype(float)
res = sys0.integrate_all_branches(all_branches, t_eval, float(P4 + 1), backend='jax')

rms = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in all_branches])
        Rk_w = np.mod(Rk, 2 * np.pi)
        Rk_w[Rk_w > np.pi] -= 2 * np.pi
        rms[idx, k] = np.sqrt(np.mean(Rk_w**2))

# ══════════════════════════════════════════════════════════════
# The spatial profile: R3 RMS as a function of position on the solenoid
# ══════════════════════════════════════════════════════════════

print('=== The solenoid as a spatial structure ===')
print(f'Period: P4 = {P4}')
print(f'Spatial scale: 1/kappa = sqrt(P4) = {1/kappa:.2f}')
print(f'  The standing wave decays over sqrt(P4) = {np.sqrt(P4):.1f} positions')
print(f'  The solenoid IS {P4} positions long')
print(f'  Ratio: P4/(1/kappa) = sqrt(P4) = {np.sqrt(P4):.1f}')
print(f'  --> The spatial extent IS the decay length × sqrt(P4)')

# The R3 spatial profile at each coprime crossing
print(f'\nR3 spatial profile (sorted by ci position):')
sorted_idx = np.argsort(cis)
for i in sorted_idx:
    ci_val = cis[i]
    r3 = rms[i, 3]
    # Analytic prediction: e^{-kappa*ci} for the dominant mode
    analytic = np.exp(-kappa * ci_val) * rms[sorted_idx[0], 3] / np.exp(-kappa * cis[sorted_idx[0]])
    bar = '#' * int(r3 * 30)
    if a5[i] == 0 and a3[i] == 1:  # quark mass sector
        gen = {0:1,3:1,1:2,4:2,2:3,5:3}[a7[i]]
        print(f'  ci={ci_val:3d} R3={r3:.4f} gen{gen} Q {bar}')

# ══════════════════════════════════════════════════════════════
# Fourier analysis: what modes make up the spatial profile?
# ══════════════════════════════════════════════════════════════

# The RMS at each coprime crossing is a function of position on Z/P4Z.
# We can decompose this into Fourier modes on the circle of period P4.
# But we only have samples at the 48 coprime positions.

# First: what does the profile look like as a function of ci?
print(f'\n=== Fourier structure of the spatial profile ===')

# Fit R3 profile to e^{-kappa*ci} (dominant mode)
# R3(ci) ≈ A * e^{-kappa*ci} + B (steady state)
from scipy.optimize import curve_fit

def exp_model(ci, A, B):
    return A * np.exp(-kappa * ci) + B

ci_float = cis.astype(float)
r3_vals = rms[:, 3]

popt, pcov = curve_fit(exp_model, ci_float, r3_vals, p0=[2.0, 0.3])
A_fit, B_fit = popt
r3_predicted = exp_model(ci_float, A_fit, B_fit)
residual = r3_vals - r3_predicted
rel_residual = residual / r3_vals

print(f'Fit R3(ci) = A*exp(-kappa*ci) + B:')
print(f'  A = {A_fit:.4f}, B = {B_fit:.4f}')
print(f'  Mean |residual/R3| = {np.mean(np.abs(rel_residual)):.4f} ({np.mean(np.abs(rel_residual))*100:.1f}%)')

# The residual structure tells us what's beyond the single exponential mode.
# If the residual is small, the spatial profile IS essentially a single mode.
# If structured, higher harmonics matter.

print(f'\n  Residual at quark generation crossings:')
gen_map = {0:1,3:1,1:2,4:2,2:3,5:3}
for idx in range(len(cis)):
    if a5[idx] == 0 and a3[idx] == 1:
        ci_val = cis[idx]
        gen = gen_map[a7[idx]]
        z2 = a7[idx] % 2
        print(f'    ci={ci_val:3d} gen{gen} Z2={z2}: R3={r3_vals[idx]:.6f} fit={r3_predicted[idx]:.6f} '
              f'resid={residual[idx]:+.6f} ({rel_residual[idx]*100:+.1f}%)')

# ══════════════════════════════════════════════════════════════
# The generation crossings sample the standing wave at specific positions
# ══════════════════════════════════════════════════════════════

print(f'\n=== How generation crossings sample the spatial wave ===')

# Quark Z2=0 crossings
ci_g2 = 11   # gen2
ci_g3 = 191  # gen3
ci_g1 = 71   # gen1

# Their positions as fractions of the solenoid period
print(f'Generation positions (fraction of P4={P4}):')
print(f'  gen2: ci={ci_g2:3d}  frac={ci_g2/P4:.4f}  phase={2*np.pi*ci_g2/P4:.4f} rad')
print(f'  gen1: ci={ci_g1:3d}  frac={ci_g1/P4:.4f}  phase={2*np.pi*ci_g1/P4:.4f} rad')
print(f'  gen3: ci={ci_g3:3d}  frac={ci_g3/P4:.4f}  phase={2*np.pi*ci_g3/P4:.4f} rad')

# The SPATIAL DECAY at each generation crossing
print(f'\nSpatial amplitude (dominant mode A*e^(-kappa*ci)):')
amp_g1 = A_fit * np.exp(-kappa * ci_g1)
amp_g2 = A_fit * np.exp(-kappa * ci_g2)
amp_g3 = A_fit * np.exp(-kappa * ci_g3)
print(f'  gen2 (ci={ci_g2}): {amp_g2:.6f}')
print(f'  gen1 (ci={ci_g1}): {amp_g1:.6f}')
print(f'  gen3 (ci={ci_g3}): {amp_g3:.6f}')

# The CP ratio in the dominant mode picture:
# CP = (A*e^{-kappa*ci_g2} + B) / (A*e^{-kappa*ci_g3} + B)
cp_model = (A_fit * np.exp(-kappa * ci_g2) + B_fit) / (A_fit * np.exp(-kappa * ci_g3) + B_fit)
print(f'\nCP ratio from spatial model: {cp_model:.4f} (actual: 6.6067)')

# ══════════════════════════════════════════════════════════════
# KEY TEST: Can the spatial profile at gen1/gen2/gen3 crossings
# directly give the mass ratios WITHOUT using exponents?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Direct spatial profile → mass ratios ===')

# The R3 RMS at each generation crossing:
r3_g1 = rms[np.where((a3==1)&(a5==0)&(a7==0))[0][0], 3]  # ci=71, gen1 Z2=0
r3_g2 = rms[np.where((a3==1)&(a5==0)&(a7==4))[0][0], 3]  # ci=11, gen2 Z2=0
r3_g3 = rms[np.where((a3==1)&(a5==0)&(a7==2))[0][0], 3]  # ci=191, gen3 Z2=0

# If mass ∝ R3^something, then mass ratios = (R3 ratios)^something
# m_b/m_s ~ gen3/gen2 in some sense, m_s/m_d ~ gen2/gen1
# But gen1 R3 < gen3 R3 at level 3 (0.586 < 0.279? No, 0.586 > 0.279)
print(f'R3 at gen crossings: gen1={r3_g1:.6f}, gen2={r3_g2:.6f}, gen3={r3_g3:.6f}')
print(f'  gen2/gen3 = {r3_g2/r3_g3:.4f} (= CP_R3 = 6.607)')
print(f'  gen1/gen3 = {r3_g1/r3_g3:.4f}')
print(f'  gen2/gen1 = {r3_g2/r3_g1:.4f}')

# What if the mass of gen_k fermion is proportional to the SPATIAL AMPLITUDE
# at that generation's crossing? Then:
# m_b ∝ R3(gen3), m_s ∝ R3(gen2), m_d ∝ R3(gen1)?  
# No — gen3 has the SMALLEST R3 but the HEAVIEST quark.
# 
# Or maybe: mass ∝ 1/R3? Then heavy gen3 has small R3 ✓
print(f'\nIf mass ∝ 1/R3 (inverse amplitude):')
print(f'  m_b/m_s ~ R3(gen2)/R3(gen3) = {r3_g2/r3_g3:.4f} = CP ratio')
print(f'  m_s/m_d ~ R3(gen1)/R3(gen3) = {r3_g1/r3_g3:.4f}')
print(f'  But m_s/m_d = 20 while R3(gen1)/R3(gen3) = {r3_g1/r3_g3:.4f}. No.')

# The CP RATIO at R3 is 6.607 and m_s/m_d = 20 = CP^{x_q}.
# The exponent x_q mediates between the spatial profile and the physical mass.
# Maybe x_q encodes HOW MANY spatial modes contribute at that crossing?

# ══════════════════════════════════════════════════════════════
# Per-branch analysis: what does the spatial profile look like
# for individual branches, not just the RMS average?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Per-branch spatial profile at R3 level ===')
# At each crossing, we have 210 branch values. The RMS averages over all branches.
# But the individual branch values carry the standing wave fine structure.

# For the quark gen crossings, look at branch-resolved R3
branches = all_branches
ci_target = [11, 71, 191]  # gen2, gen1, gen3
for ci_val in ci_target:
    ci_idx = np.where(cis == ci_val)[0][0]
    R3_branches = np.array([res[br][ci_idx, 3] for br in branches])
    R3_wrapped = np.mod(R3_branches, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    n_wrapped = np.sum(np.abs(R3_branches) > np.pi)
    frac_wrapped = n_wrapped / len(branches)
    print(f'  ci={ci_val:3d}: mean={np.mean(R3_wrapped):+.4f} std={np.std(R3_wrapped):.4f} '
          f'|raw|_max={np.max(np.abs(R3_branches)):.2f} wrapped={frac_wrapped*100:.1f}%')

# The WRAPPING FRACTION is the key variable (NB153).
# At ci=11 (gen2): most branches have |R3| > pi → heavy wrapping
# At ci=191 (gen3): few/no branches wrap → mostly coherent
# The CP ratio encodes wrapping vs non-wrapping regime difference.
# And the EXPONENT x_q comes from how wrapping translates to mass.

print(f'\n=== Wrapping as the spatial mechanism ===')
print(f'The solenoid is a spatial structure. At each position (crossing):')
print(f'  - Some branches have wrapped (|R3| > pi): large covering misalignment')  
print(f'  - Some haven\'t: small misalignment')
print(f'  - The FRACTION of wrapped branches varies across the solenoid')
print(f'  - THIS spatial variation of wrapping fraction IS the standing wave')
print(f'  - Mass comes from how this wave is sampled at generation crossings')

# Check if wrapping fraction at each crossing correlates with mass
print(f'\nWrapping fraction at a5=0 quark crossings:')
gen_map = {0:1,3:1,1:2,4:2,2:3,5:3}
wrap_data = []
for idx in range(len(cis)):
    if a5[idx] != 0 or a3[idx] != 1:
        continue
    ci_val = cis[idx]
    R3_branches = np.array([res[br][idx, 3] for br in branches])
    frac_w = np.sum(np.abs(R3_branches) > np.pi) / len(branches)
    gen = gen_map[a7[idx]]
    z2 = a7[idx] % 2
    wrap_data.append((ci_val, gen, z2, frac_w, rms[idx, 3]))
    print(f'  ci={ci_val:3d} gen{gen} Z2={z2}: wrapped={frac_w*100:5.1f}%  RMS_R3={rms[idx,3]:.4f}')

# The generation stepping in wrapping-fraction space:
print(f'\nWrapping fraction at Z2=0 quark gen crossings:')
w_g1 = [d[3] for d in wrap_data if d[1]==1 and d[2]==0][0]  # gen1 Z2=0
w_g2 = [d[3] for d in wrap_data if d[1]==2 and d[2]==0][0]  # gen2 Z2=0
w_g3 = [d[3] for d in wrap_data if d[1]==3 and d[2]==0][0]  # gen3 Z2=0
print(f'  gen2 (ci=11):  {w_g2*100:.1f}%')
print(f'  gen1 (ci=71):  {w_g1*100:.1f}%')
print(f'  gen3 (ci=191): {w_g3*100:.1f}%')
print(f'  gen2-gen3 difference: {(w_g2-w_g3)*100:.1f}pp')
print(f'  gen1-gen3 difference: {(w_g1-w_g3)*100:.1f}pp')

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.46s
=== The solenoid as a spatial structure ===
Period: P4 = 210
Spatial scale: 1/kappa = sqrt(P4) = 14.49
  The standing wave decays over sqrt(P4) = 14.5 positions
  The solenoid IS 210 positions long
  Ratio: P4/(1/kappa) = sqrt(P4) = 14.5
  --> The spatial extent IS the decay length × sqrt(P4)

R3 spatial profile (sorted by ci position):
  ci= 11 R3=1.8465 gen2 Q #######################################################
  ci= 41 R3=2.0761 gen1 Q ##############################################################
  ci= 71 R3=0.5858 gen1 Q #################
  ci=101 R3=0.3308 gen2 Q #########
  ci=131 R3=0.2880 gen3 Q ########
  ci=191 R3=0.2795 gen3 Q ########

=== Fourier structure of the spatial profile ===
Fit R3(ci) = A*exp(-kappa*ci) + B:
  A = 2.4585, B = 0.4899
  Mean |residual/R3| = 1.4378 (143.8%)

  Residual at quark generation crossings:
    ci= 11 gen2 Z2=0: R3=1.846494 fit=1.640726 resid=+0.205767 (+11.1%)
    ci

In [1]:
# S1 — What determines the scaling factors r_bs and r_tc?
# 
# Since r = ln(mass_ratio) / ln(m_s/m_d), and m_s/m_d = CP_R3^{x_q},
# we have r = ln(mass_ratio) / (x_q * ln(CP_R3)).
# This is circular unless we can compute r from the solenoid directly.
#
# Strategy: look at the GENERATION STRUCTURE of the crossings.
# The CP pair compares gen2 (ci=11) to gen3 (ci=191).
# Other generations live at other crossings.
# The mass ratios might come from the RELATIVE POSITIONS of these crossings.

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi
lambda_P4 = reduce(_lcm, [p - 1 for p in primes])
phi_P4 = 48

P = [1]
for p in primes:
    P.append(P[-1] * p)

cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

# The quark a5=0 crossings by generation and Z2 parity
print('=== Generation structure of a5=0 quark crossings ===')
print(f'{"ci":>4}  {"a3":>3}  {"a7":>3}  {"Z3":>3}  {"Z2":>3}  {"gen":>4}  {"e^(-kappa*ci)":>14}')
print('-' * 50)
gen_map = {0: 1, 3: 1, 1: 2, 4: 2, 2: 3, 5: 3}
for idx in range(len(cis)):
    if a5[idx] != 0 or a3[idx] != 1:  # quark sector only
        continue
    ci_val = cis[idx]
    z3 = a7[idx] % 3
    z2 = a7[idx] % 2
    gen = gen_map[a7[idx]]
    decay = np.exp(-kappa * ci_val)
    print(f'{ci_val:4d}  {a3[idx]:3d}  {a7[idx]:3d}  {z3:3d}  {z2:3d}  {gen:4d}  {decay:14.6e}')

# The CP pair uses ci=11 (gen2, Z2=0) and ci=191 (gen3, Z2=0)
# The other gen crossings:
# Gen1 Z2=0: ci=71 (a7=0)   
# Gen1 Z2=1: ci=41 (a7=3)
# Gen2 Z2=0: ci=11 (a7=4)  <- CP g1
# Gen2 Z2=1: ci=101 (a7=1)
# Gen3 Z2=0: ci=191 (a7=2) <- CP g2
# Gen3 Z2=1: ci=131 (a7=5)

# The EXPONENTIAL DECAY at each crossing:
ci_gens = {
    'gen1_Z2=0': 71, 'gen1_Z2=1': 41,
    'gen2_Z2=0': 11, 'gen2_Z2=1': 101,
    'gen3_Z2=0': 191, 'gen3_Z2=1': 131,
}

print(f'\nDecay factors e^(-kappa*ci):')
for name, ci_val in ci_gens.items():
    print(f'  {name:>12}: ci={ci_val:3d}  decay={np.exp(-kappa*ci_val):.6e}')

# The mass ratio is related to the RATIO of decay factors.
# For the CP pair: decay(11)/decay(191) = e^{kappa*(191-11)} = e^{kappa*180}
print(f'\n=== Decay RATIOS between generations ===')
print(f'CP pair (gen2/gen3): e^(kappa*{191-11}) = e^({kappa*(191-11):.4f}) = {np.exp(kappa*(191-11)):.4f}')

# But the mass ratio is NOT the decay ratio — it's CP_R3^x_q.
# The CP ratio at R3 = 6.607. 
# The decay ratio = e^{kappa*180} = e^{12.42} = 248,000.
# These are vastly different! The CP ratio comes from the WINDOWED RMS, not raw decay.

# Let's look at the CROSSING POSITION DIFFERENCES between generations:
print(f'\n=== Crossing position structure ===')
ci_g1_z0, ci_g2_z0, ci_g3_z0 = 71, 11, 191  # Z2=0 crossings
ci_g1_z1, ci_g2_z1, ci_g3_z1 = 41, 101, 131  # Z2=1 crossings

# Distances in ci-space (mod P4=210):
def ci_dist(a, b):
    d = abs(a - b)
    return min(d, P4 - d)  # shortest distance on the circle

print(f'Z2=0 quark crossings: gen1={ci_g1_z0}, gen2={ci_g2_z0}, gen3={ci_g3_z0}')
print(f'  gen1-gen2: {ci_dist(ci_g1_z0, ci_g2_z0)} (mod 210)')
print(f'  gen2-gen3: {ci_dist(ci_g2_z0, ci_g3_z0)} (mod 210)')
print(f'  gen1-gen3: {ci_dist(ci_g1_z0, ci_g3_z0)} (mod 210)')

print(f'\nZ2=1 quark crossings: gen1={ci_g1_z1}, gen2={ci_g2_z1}, gen3={ci_g3_z1}')
print(f'  gen1-gen2: {ci_dist(ci_g1_z1, ci_g2_z1)} (mod 210)')
print(f'  gen2-gen3: {ci_dist(ci_g2_z1, ci_g3_z1)} (mod 210)')
print(f'  gen1-gen3: {ci_dist(ci_g1_z1, ci_g3_z1)} (mod 210)')

# The ACTUAL ci values come from CRT: ci = a3*a + a5*b + a7*c (mod 210)
# Let's verify by reconstructing from CRT
print(f'\n=== CRT reconstruction ===')
# Z*_210 = Z2 x Z4 x Z6, with a3 in Z2, a5 in Z4, a7 in Z6
# CRT gives: for a5=0, the ci depends on a3 and a7 only.
# The CRT map sends (a3, a5, a7) -> ci such that:
#   ci mod 3 = {0,1,2} based on a3
#   ci mod 5 = {0,1,2,3,4} based on a5
#   ci mod 7 = {0,1,2,3,4,5,6} based on a7
# Actually the labels a3,a5,a7 are discrete log indices, not residues directly.

# What matters: the ci positions are determined by the CRT structure.
# The SPACINGS between generation crossings encode the Z6 → Z3 decomposition.

# Let me look at the generation structure more abstractly.
# Within the CP pair (gen2 Z2=0 vs gen3 Z2=0), the CP ratio encodes
# one specific asymmetry. The MASS uses this asymmetry raised to an exponent.
# 
# For the 1->2 step: x_q gives m_s/m_d = 20.0
# For the 2->3 step: x_q * r gives m_b/m_s = 44.75 where r = 1.269
# For the up-type: x_q * r' gives m_t/m_c = 135.8 where r' = 1.639
#
# What if r and r' come from the Z2 structure within Z6?

# The CP pair lives in Z2 coset 0 (a7 even → a7 mod 2 = 0)
# The 1->2 step compares gen1 to gen2 WITHIN Z2 coset 0: ci=71 vs ci=11
# The 2->3 step compares gen2 to gen3 WITHIN Z2 coset 0: ci=11 vs ci=191 (= the CP pair itself)
# 
# But the mass exponent for the 1->2 step (x_q) was measured from the CP pair
# (which IS the gen2/gen3 comparison). So x_q is the gen2→gen3 eigenvalue.
# When we use it for the gen1→2 step, we're applying the gen2→3 eigenvalue
# to a gen1→2 comparison. This ONLY works if the CP ratio at gen1→2 is related
# to the CP ratio at gen2→3 by a specific factor.

# Check: what's the RMS ratio at gen1 vs gen2 crossings?
sys0 = SolenoidSystem(primes=primes)
all_branches = sys0.all_branches()

from solenoid_jax import warmup
warmup()
t_eval = cis.astype(float)
T_max = float(P4 + 1)
res = sys0.integrate_all_branches(all_branches, t_eval, T_max, backend='jax')

rms = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in all_branches])
        Rk_w = np.mod(Rk, 2 * np.pi)
        Rk_w[Rk_w > np.pi] -= 2 * np.pi
        rms[idx, k] = np.sqrt(np.mean(Rk_w**2))

# Get indices for each quark generation crossing
def get_idx(a3v, a7v):
    mask = (a3 == a3v) & (a5 == 0) & (a7 == a7v)
    return np.where(mask)[0][0]

# All six quark crossings (a3=1, a5=0)
q_crossings = {}
for a7v in range(6):
    idx = get_idx(1, a7v)
    q_crossings[a7v] = {'idx': idx, 'ci': cis[idx], 'rms': rms[idx]}

print(f'\n=== All quark (a3=1, a5=0) RMS by a7 ===')
print(f'{"a7":>3} {"ci":>4} {"gen":>4} {"Z2":>3}  {"R0":>10} {"R1":>10} {"R2":>10} {"R3":>10}')
for a7v in [0, 3, 4, 1, 2, 5]:  # sorted by gen then Z2
    d = q_crossings[a7v]
    gen = gen_map[a7v]
    z2 = a7v % 2
    r = d['rms']
    print(f'{a7v:3d} {d["ci"]:4d} {gen:4d} {z2:3d}  {r[0]:10.6f} {r[1]:10.6f} {r[2]:10.6f} {r[3]:10.6f}')

# Now compute CROSS-GENERATION ratios at each level
print(f'\n=== Cross-generation RMS ratios (Z2=0 coset, level R3) ===')
r_g1 = q_crossings[0]['rms']   # gen1 Z2=0 (ci=71)
r_g2 = q_crossings[4]['rms']   # gen2 Z2=0 (ci=11) = CP g1
r_g3 = q_crossings[2]['rms']   # gen3 Z2=0 (ci=191) = CP g2

print(f'gen2/gen3 (CP pair):   R3 = {r_g2[3]/r_g3[3]:.6f}  (this is CP_R3 = 6.607)')
print(f'gen1/gen2:             R3 = {r_g1[3]/r_g2[3]:.6f}')
print(f'gen1/gen3:             R3 = {r_g1[3]/r_g3[3]:.6f}')

# The mass ratios use:
# m_s/m_d ~ CP_R3^{x_q} = (gen2/gen3)^{x_q} = 6.607^1.587 = 20.0
# m_b/m_s ~ CP_R3^{2.013} = 6.607^2.013 = 44.6
# These use the SAME base (gen2/gen3 ratio) with different exponents.
# But physically, m_b/m_s compares gen3 to gen2, and m_s/m_d compares gen2 to gen1.
# They both use the gen2/gen3 CP ratio as the base.
# The exponent encodes HOW MUCH of the gen2/gen3 asymmetry each mass ratio "sees".

# What if the ratio r_bs = 1.269 is determined by the gen1/gen2 RMS ratio
# compared to the gen2/gen3 RMS ratio?
print(f'\n=== Testing: does the cross-gen RMS structure determine r? ===')
for k in range(4):
    r23 = r_g2[k] / r_g3[k]  # gen2/gen3
    r12 = r_g1[k] / r_g2[k]  # gen1/gen2
    r13 = r_g1[k] / r_g3[k]  # gen1/gen3
    # m_s/m_d = r23^{x_q(k)}. m_b/m_s = r23^{x_q(k) * r_bs}.
    # But m_b/m_s could also come from a DIFFERENT cross-gen comparison.
    # What if m_b/m_s = (gen1/gen3)^{x_q(k)} / (gen1/gen2)^{x_q(k)} ?
    # = (r13/r12)^{x_q(k)} = r23^{x_q(k)} = m_s/m_d. That's circular.
    
    # Or: m_b/m_s involves gen3→gen2 at level k, amplified by some factor from
    # the gen1 crossing position.
    
    # Let me just compute all possible simple ratios and see what matches.
    if r23 > 1.001:
        x_q_k = np.log(20.0) / np.log(r23)  # factored exponent at this level
        print(f'  R{k}: gen2/gen3={r23:.4f}  gen1/gen2={r12:.4f}  gen1/gen3={r13:.4f}')
        print(f'       x_q(R{k})={x_q_k:.4f}  ln(gen1/gen2)/ln(gen2/gen3)={np.log(r12)/np.log(r23):.4f}')

# The KEY ratio: ln(gen1/gen2) / ln(gen2/gen3) at each level
print(f'\n=== ln(gen1/gen2) / ln(gen2/gen3) at each level ===')
for k in range(4):
    r23 = r_g2[k] / r_g3[k]
    r12 = r_g1[k] / r_g2[k]
    if r23 > 1.001 and r12 > 0:
        ratio = np.log(abs(r12)) / np.log(r23) if r12 > 0.001 else float('nan')
        print(f'  R{k}: {ratio:.6f}')

# Also check: ln(gen1/gen3) / ln(gen2/gen3)
print(f'\n=== ln(gen1/gen3) / ln(gen2/gen3) at each level ===')
for k in range(4):
    r23 = r_g2[k] / r_g3[k]
    r13 = r_g1[k] / r_g3[k]
    if r23 > 1.001 and r13 > 0.001:
        ratio = np.log(r13) / np.log(r23)
        print(f'  R{k}: {ratio:.6f}')
        # This ratio tells us: how many "gen2→3 steps" does the gen1→3 span equal?
        # If r13 = r23^n, then the gen1→3 span is n "gen2→3 steps".

# And the Z2=1 crossings?
print(f'\n=== Z2=1 coset cross-gen ratios ===')
r_g1_z1 = q_crossings[3]['rms']  # gen1 Z2=1 (ci=41)
r_g2_z1 = q_crossings[1]['rms']  # gen2 Z2=1 (ci=101)
r_g3_z1 = q_crossings[5]['rms']  # gen3 Z2=1 (ci=131)

for k in range(4):
    r23 = r_g2_z1[k] / r_g3_z1[k]
    r12 = r_g1_z1[k] / r_g2_z1[k]
    r13 = r_g1_z1[k] / r_g3_z1[k]
    if abs(r23) > 0.001:
        print(f'  R{k}: gen2/gen3={r23:.4f}  gen1/gen2={r12:.4f}  gen1/gen3={r13:.4f}')

=== Generation structure of a5=0 quark crossings ===
  ci   a3   a7   Z3   Z2   gen   e^(-kappa*ci)
--------------------------------------------------
  11    1    4    1    0     2    4.681006e-01
  41    1    3    0    1     1    5.905602e-02
  71    1    0    0    0     1    7.450565e-03
 101    1    1    1    1     2    9.399704e-04
 131    1    5    2    1     3    1.185876e-04
 191    1    2    2    0     3    1.887510e-06

Decay factors e^(-kappa*ci):
     gen1_Z2=0: ci= 71  decay=7.450565e-03
     gen1_Z2=1: ci= 41  decay=5.905602e-02
     gen2_Z2=0: ci= 11  decay=4.681006e-01
     gen2_Z2=1: ci=101  decay=9.399704e-04
     gen3_Z2=0: ci=191  decay=1.887510e-06
     gen3_Z2=1: ci=131  decay=1.185876e-04

=== Decay RATIOS between generations ===
CP pair (gen2/gen3): e^(kappa*180) = e^(12.4212) = 247999.0187

=== Crossing position structure ===
Z2=0 quark crossings: gen1=71, gen2=11, gen3=191
  gen1-gen2: 60 (mod 210)
  gen2-gen3: 30 (mod 210)
  gen1-gen3: 90 (mod 210)

Z2=1 quar

## Key Finding from S0

The ratio (needed exponent)/(factored exponent) is **exactly level-independent** (spread = 0):
- m_b/m_s: scaling = 1.2688 at ALL four levels
- m_t/m_c: scaling = 1.6394 at ALL four levels

This means the "2" at R3 and "4/3" at R2 are NOT fundamental. They are:
- At R3: x_q × 1.2688 = 2.013 ≈ 2 (coincidence with integer)
- At R2: x_q × 1.6394 / cl(R3→R2) = 1.333 ≈ 4/3 (coincidence with rational)

The REAL unknowns are two universal scaling factors: r_bs = 1.2688 and r_tc = 1.6394.

**All mass ratios have the form**: mass_ratio = CP_Rk^{x_q × r × cl_inv(Rk)}
where r = 1 for 1→2 gen, r = r_bs for 2→3 down, r = r_tc for 2→3 up.

What determines r_bs and r_tc?

# NB156 — Why These Exponents?

**Goal**: Understand, not just verify, the mass pipeline. Three open questions:

1. **Why T=211?** Why does one primorial period capture all mass physics?
2. **Why 2 and 4/3?** The 2->3 generation exponents are empirically matched to PDG.
   Can we DERIVE them from the cascade or the covering topology?
3. **Why the qualitative split?** The 1->2 gen step uses a dynamical eigenvalue (x_q=1.587).
   The 2->3 gen step uses clean rationals (2, 4/3). Why are these different mechanisms?

In [2]:
# S0 — Setup, cascade, CP extraction, and T-dependence study
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'

import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm

from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)  # 210
kappa = 1.0 / np.sqrt(P4)
epsilon = kappa
omega = 2 * np.pi
lambda_P4 = reduce(_lcm, [p - 1 for p in primes])  # 12
phi_P4 = 48  # phi(210)

P = [1]
for p in primes:
    P.append(P[-1] * p)
# P = [1, 2, 6, 30, 210]

# ══════════════════════════════════════════════════════════════
# QUESTION 1: Why T=211? How fast does the transient die?
# ══════════════════════════════════════════════════════════════

# The cascade is overdamped: kappa = 1/sqrt(210) = 0.06901.
# The R0 analytic solution (NB138) gives R0(t) ~ e^{-kappa*t}.
# After one period: e^{-kappa*P4} = e^{-sqrt(210)} ≈ 5e-7.
print('=== QUESTION 1: Why T=211 (one primorial period)? ===')
print(f'kappa = 1/sqrt({P4}) = {kappa:.6f}')
print(f'Decay per crossing: e^(-kappa) = {np.exp(-kappa):.6f} ({(1-np.exp(-kappa))*100:.2f}% per step)')
print(f'Decay after 1 period (P4={P4}): e^(-sqrt({P4})) = {np.exp(-np.sqrt(P4)):.2e}')
print(f'Decay half-life: ln(2)/kappa = {np.log(2)/kappa:.1f} crossings')
print(f'Number of coprime crossings per period: {phi_P4}')
print(f'Time constants in one period: P4*kappa = sqrt(P4) = {np.sqrt(P4):.2f}')
print(f'  So one period = {np.sqrt(P4):.1f} e-folding times.')
print(f'  Residual transient: e^(-{np.sqrt(P4):.1f}) = {np.exp(-np.sqrt(P4)):.2e}')

# The KEY point: kappa*P4 = sqrt(P4) = 14.49.
# One period contains 14.5 time constants. That's why window 0 captures everything.
# It's not a choice — it's BUILT INTO the relationship kappa = 1/sqrt(P4):
#   kappa * P4 = P4/sqrt(P4) = sqrt(P4)
# The number of e-folding times per period IS sqrt(P4). For any solenoid.

print(f'\n--- This is structural, not accidental ---')
print(f'For ANY (p1,...,pk)-solenoid with kappa = 1/sqrt(Pk):')
print(f'  e-foldings per period = sqrt(Pk)')
print(f'  sqrt(P1)=sqrt(2)   = {np.sqrt(2):.2f}  (barely enough)')
print(f'  sqrt(P2)=sqrt(6)   = {np.sqrt(6):.2f}  (marginal)')
print(f'  sqrt(P3)=sqrt(30)  = {np.sqrt(30):.2f} (good)')
print(f'  sqrt(P4)=sqrt(210) = {np.sqrt(210):.2f} (14+ time constants, overkill)')

# Now verify: do CP ratios ACTUALLY differ between windows?
sys0 = SolenoidSystem(primes=primes)
all_branches = sys0.all_branches()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()

def compute_cp_at_window(window_idx):
    """Compute CP ratios at coprime crossings within a specific window."""
    t_eval = (cis + window_idx * P4).astype(float)
    T_max = float((window_idx + 1) * P4 + 1)
    res = sys0.integrate_all_branches(all_branches, t_eval, T_max, backend='jax')
    
    n_cross = len(cis)
    rms = np.zeros((n_cross, 4))
    for idx in range(n_cross):
        for k in range(4):
            Rk = np.array([res[br][idx, k] for br in all_branches])
            Rk_w = np.mod(Rk, 2 * np.pi)
            Rk_w[Rk_w > np.pi] -= 2 * np.pi
            rms[idx, k] = np.sqrt(np.mean(Rk_w**2))
    
    cp = {}
    for name, (ch_a3, a7_g1, a7_g2) in CP_PAIRS.items():
        g1_mask = (a3 == ch_a3) & (a5 == 0) & (a7 == a7_g1)
        g2_mask = (a3 == ch_a3) & (a5 == 0) & (a7 == a7_g2)
        i1, i2 = np.where(g1_mask)[0][0], np.where(g2_mask)[0][0]
        cp[name] = rms[i1] / rms[i2]
    return cp, rms

# Windows 0, 1, 2
cp_w0, rms_w0 = compute_cp_at_window(0)
cp_w1, rms_w1 = compute_cp_at_window(1)
cp_w2, rms_w2 = compute_cp_at_window(2)

print(f'\n--- CP ratios by window and level ---')
print(f'{"":>16} {"R0":>10} {"R1":>10} {"R2":>10} {"R3":>10}')
for w_label, cp_w in [('W0', cp_w0), ('W1', cp_w1), ('W2', cp_w2)]:
    for ch in ['QUARK', 'LEPTON']:
        vals = [cp_w[ch][k] for k in range(4)]
        print(f'{w_label+" "+ch:>16} {vals[0]:10.4f} {vals[1]:10.4f} {vals[2]:10.4f} {vals[3]:10.4f}')

print(f'\n--- Window 1 vs Window 0 (ratio) ---')
for ch in ['QUARK', 'LEPTON']:
    ratios = [cp_w1[ch][k] / cp_w0[ch][k] for k in range(4)]
    print(f'{ch:>10}: {ratios[0]:.6f} {ratios[1]:.6f} {ratios[2]:.6f} {ratios[3]:.6f}')

# ══════════════════════════════════════════════════════════════
# QUESTION 2: The exponents. What determines them?
# ══════════════════════════════════════════════════════════════

print(f'\n\n=== QUESTION 2: Where do the exponents come from? ===')

cpQ = cp_w0['QUARK']
cpL = cp_w0['LEPTON']

# The cross-level structure: how CP ratios relate across levels
print(f'Cross-level ratios (log ratios):')
ln_cpQ = {k: np.log(cpQ[k]) for k in range(4)}
ln_cpL = {k: np.log(cpL[k]) for k in range(4)}

print(f'  Quark:  ln(CP_R0)/ln(CP_R3) = {ln_cpQ[0]/ln_cpQ[3]:.6f}')
print(f'  Quark:  ln(CP_R1)/ln(CP_R3) = {ln_cpQ[1]/ln_cpQ[3]:.6f}')
print(f'  Quark:  ln(CP_R2)/ln(CP_R3) = {ln_cpQ[2]/ln_cpQ[3]:.6f}')
print(f'  Lepton: ln(CP_R0)/ln(CP_R3) = {ln_cpL[0]/ln_cpL[3]:.6f}')
print(f'  Lepton: ln(CP_R1)/ln(CP_R3) = {ln_cpL[1]/ln_cpL[3]:.6f}')
print(f'  Lepton: ln(CP_R2)/ln(CP_R3) = {ln_cpL[2]/ln_cpL[3]:.6f}')

# The factored architecture says CP_Rk^{x_Rk} = same value for all k.
# So x_Rk = x_R3 * ln(CP_R3)/ln(CP_Rk) = x_R3 / cross_level(R3->Rk)
x_q = 1.5866463961
x_l = 3.0003758562
print(f'\nFactored exponents (x_q * cross_level_inv):')
for k in range(4):
    x_q_k = x_q * ln_cpQ[3] / ln_cpQ[k]
    x_l_k = x_l * ln_cpL[3] / ln_cpL[k]
    print(f'  Level {k}: x_q_R{k} = {x_q_k:.6f},  x_l_R{k} = {x_l_k:.6f}')

# KEY: These factored exponents give m_s/m_d at EVERY level. 
# But the inter-gen ratios use DIFFERENT exponents at different levels.
# The question is: what IS the exponent for the 2->3 step?

print(f'\n--- The 2->3 generation exponents ---')
# At level R3: the fitted exponent for m_b/m_s is 2.013
# At level R2: the fitted exponent for m_t/m_c is 1.333 = 4/3
# These are NOT the factored x_q projected to those levels.

# What are the factored exponents at those levels?
x_q_at_R3 = x_q  # 1.587 (this gives m_s/m_d)
x_q_at_R2 = x_q * ln_cpQ[3] / ln_cpQ[2]  # x_q projected to R2

print(f'  Factored x_q at R3: {x_q_at_R3:.6f} (gives m_s/m_d = {cpQ[3]**x_q_at_R3:.2f})')
print(f'  Factored x_q at R2: {x_q_at_R2:.6f} (gives m_s/m_d = {cpQ[2]**x_q_at_R2:.2f})')
print(f'  But m_b/m_s needs exponent 2.013 at R3 (vs factored {x_q_at_R3:.4f})')
print(f'  And m_t/m_c needs exponent 1.333 at R2 (vs factored {x_q_at_R2:.4f})')

# The RATIO of the 2->3 exponent to the factored exponent:
ratio_bs = 2.013 / x_q_at_R3
ratio_tc = (4/3) / x_q_at_R2
print(f'\n  Ratio (2->3 exp)/(factored exp):')
print(f'    m_b/m_s at R3: 2.013 / {x_q_at_R3:.4f} = {ratio_bs:.6f}')
print(f'    m_t/m_c at R2: (4/3) / {x_q_at_R2:.4f} = {ratio_tc:.6f}')

# Are these the same? That would mean the 2->3 step is a UNIFORM
# rescaling of the 1->2 step.
print(f'    Ratio of ratios: {ratio_bs / ratio_tc:.6f}')

# ══════════════════════════════════════════════════════════════
# Explore: what makes the exponent at each level?
# ══════════════════════════════════════════════════════════════
print(f'\n\n--- Exploring the cross-level structure ---')

# The CP ratio at each level comes from the RMS ratio of g1 vs g2.
# Let's look at the INDIVIDUAL RMS values, not just ratios.
q_g1_idx = np.where((a3 == 1) & (a5 == 0) & (a7 == 4))[0][0]
q_g2_idx = np.where((a3 == 1) & (a5 == 0) & (a7 == 2))[0][0]

print(f'Quark g1 (ci={cis[q_g1_idx]}): RMS = {rms_w0[q_g1_idx]}')
print(f'Quark g2 (ci={cis[q_g2_idx]}): RMS = {rms_w0[q_g2_idx]}')

# The g1 RMS at each level shows the cascade amplification
print(f'\ng1 RMS by level: {rms_w0[q_g1_idx]}')
print(f'g2 RMS by level: {rms_w0[q_g2_idx]}')

# What fraction of each level's RMS comes from the covering map amplification?
# R_k = p_{k+1} * R_{k+1} - theta_k -> at steady state, R_k ~ p_{k+1} * R_{k+1}
# So CP_Rk should scale as p_{k+1} * CP_{Rk+1} (approximately)
print(f'\nCovering amplification test: CP_Rk / CP_R(k+1) vs p_(k+2)')
for k in range(3):
    amp_q = cpQ[k] / cpQ[k+1]
    amp_l = cpL[k] / cpL[k+1]
    p_next = primes[k+1] if k+1 < 4 else '?'
    print(f'  CP_R{k}/CP_R{k+1}: Quark={amp_q:.4f}, Lepton={amp_l:.4f}  (p_{k+2}={primes[k+1]})')

# The CP cross-level structure encodes the covering map cascade.
# Each level amplifies by approximately p_{k+1}.
# But the EXPONENT structure is different: x_q is constant across levels
# for the 1->2 step, but the 2->3 step uses different exponents at each level.

print(f'\n\n=== QUESTION 3: Why the qualitative split? ===')
print(f'The 1->2 gen step: SAME dynamical exponent x_q at every level')
print(f'  m_s/m_d = CP_R3^x_q = {cpQ[3]**x_q:.2f}')
print(f'  m_s/m_d = CP_R2^(x_q*cl) = {cpQ[2]**(x_q*ln_cpQ[3]/ln_cpQ[2]):.2f} (same value)')
print(f'  m_s/m_d = CP_R1^(x_q*cl) = {cpQ[1]**(x_q*ln_cpQ[3]/ln_cpQ[1]):.2f} (same value)')

print(f'\nBut for the 2->3 step:')
print(f'  m_b/m_s = 44.75 requires DIFFERENT exponents at different levels:')
for k in range(4):
    if cpQ[k] > 1.001:
        x_needed = np.log(44.75) / np.log(cpQ[k])
        x_factored = x_q * ln_cpQ[3] / ln_cpQ[k]
        print(f'    R{k}: needed={x_needed:.4f}, factored={x_factored:.4f}, ratio={x_needed/x_factored:.4f}')

print(f'\n  m_t/m_c = 135.8 requires:')
for k in range(4):
    if cpQ[k] > 1.001:
        x_needed = np.log(135.8) / np.log(cpQ[k])
        x_factored = x_q * ln_cpQ[3] / ln_cpQ[k]
        print(f'    R{k}: needed={x_needed:.4f}, factored={x_factored:.4f}, ratio={x_needed/x_factored:.4f}')

# The ratio of (needed/factored) should be the same at ALL levels if the
# 2->3 step is a simple rescaling. Let's check.
print(f'\nRatio (needed/factored) for m_b/m_s:')
r_bs = []
for k in range(4):
    if cpQ[k] > 1.001:
        x_needed = np.log(44.75) / np.log(cpQ[k])
        x_factored = x_q * ln_cpQ[3] / ln_cpQ[k]
        r = x_needed / x_factored
        r_bs.append(r)
        print(f'  R{k}: {r:.6f}')
print(f'  Spread: {max(r_bs)-min(r_bs):.6f}')
print(f'  Mean: {np.mean(r_bs):.6f}')

print(f'\nRatio (needed/factored) for m_t/m_c:')
r_tc = []
for k in range(4):
    if cpQ[k] > 1.001:
        x_needed = np.log(135.8) / np.log(cpQ[k])
        x_factored = x_q * ln_cpQ[3] / ln_cpQ[k]
        r = x_needed / x_factored
        r_tc.append(r)
        print(f'  R{k}: {r:.6f}')
print(f'  Spread: {max(r_tc)-min(r_tc):.6f}')
print(f'  Mean: {np.mean(r_tc):.6f}')

# If the ratio is CONSTANT across levels, then the 2->3 step is just
# a uniform multiplicative factor on the exponent.
# If it VARIES, then there's level-dependent structure.

# Also: what algebraic expressions match these ratios?
print(f'\n--- Algebraic candidates for the scaling factor ---')
r_bs_mean = np.mean(r_bs)
r_tc_mean = np.mean(r_tc)
print(f'  m_b/m_s scaling: {r_bs_mean:.6f}')
print(f'  m_t/m_c scaling: {r_tc_mean:.6f}')
candidates = [
    (p1, 'p1'), (p2, 'p2'), (p3/p4, 'p3/p4'), (p4/p3, 'p4/p3'),
    (4/3, '4/3'), (2, '2'), (lambda_P4/(2*np.pi), 'lam/2pi'),
    (np.sqrt(p1), 'sqrt(p1)'), (np.sqrt(p4/p3), 'sqrt(p4/p3)'),
    (phi_P4/P4, 'phi/P4'), (lambda_P4/phi_P4, 'lam/phi'),
]
for val, name in candidates:
    d_bs = abs(r_bs_mean - val) / r_bs_mean * 100
    d_tc = abs(r_tc_mean - val) / r_tc_mean * 100
    marker = ''
    if d_bs < 2 or d_tc < 2:
        marker = ' ***'
    print(f'  {name:>10} = {val:.6f}  |  dev(b/s)={d_bs:5.2f}%  dev(t/c)={d_tc:5.2f}%{marker}')

=== QUESTION 1: Why T=211 (one primorial period)? ===
kappa = 1/sqrt(210) = 0.069007
Decay per crossing: e^(-kappa) = 0.933321 (6.67% per step)
Decay after 1 period (P4=210): e^(-sqrt(210)) = 5.09e-07
Decay half-life: ln(2)/kappa = 10.0 crossings
Number of coprime crossings per period: 48
Time constants in one period: P4*kappa = sqrt(P4) = 14.49
  So one period = 14.5 e-folding times.
  Residual transient: e^(-14.5) = 5.09e-07

--- This is structural, not accidental ---
For ANY (p1,...,pk)-solenoid with kappa = 1/sqrt(Pk):
  e-foldings per period = sqrt(Pk)
  sqrt(P1)=sqrt(2)   = 1.41  (barely enough)
  sqrt(P2)=sqrt(6)   = 2.45  (marginal)
  sqrt(P3)=sqrt(30)  = 5.48 (good)
  sqrt(P4)=sqrt(210) = 14.49 (14+ time constants, overkill)
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.62s
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=421.0 — 3.67s
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=631.0 — 5.10s

--- CP ratios by window and level ---
       